In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
import gradio as gr

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Final_Augmented_dataset_Diseases_and_Symptoms.csv to Final_Augmented_dataset_Diseases_and_Symptoms.csv


In [3]:
df = pd.read_csv("Final_Augmented_dataset_Diseases_and_Symptoms.csv")
df.head()

,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [4]:
X = df.drop("diseases", axis=1)
y = df["diseases"]

X.columns = X.columns.str.strip().str.lower()

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

disease_mapping = dict(zip(le.classes_, le.transform(le.classes_)))

print("Disease Mapping:")
for disease, num in disease_mapping.items():
    print(f"{num} -> {disease}")

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

Disease Mapping:
0 -> abdominal aortic aneurysm
1 -> abdominal hernia
2 -> abscess of nose
3 -> abscess of the lung
4 -> abscess of the pharynx
5 -> acanthosis nigricans
6 -> acariasis
7 -> achalasia
8 -> acne
9 -> actinic keratosis
10 -> acute bronchiolitis
11 -> acute bronchitis
12 -> acute bronchospasm
13 -> acute fatty liver of pregnancy (aflp)
14 -> acute glaucoma
15 -> acute kidney injury
16 -> acute otitis media
17 -> acute pancreatitis
18 -> acute respiratory distress syndrome (ards)
19 -> acute sinusitis
20 -> acute stress reaction
21 -> adhesive capsulitis of the shoulder
22 -> adjustment reaction
23 -> adrenal adenoma
24 -> adrenal cancer
25 -> alcohol abuse
26 -> alcohol intoxication
27 -> alcohol withdrawal
28 -> alcoholic liver disease
29 -> allergy
30 -> allergy to animals
31 -> alopecia
32 -> alzheimer disease
33 -> amblyopia
34 -> amyloidosis
35 -> amyotrophic lateral sclerosis (als)
36 -> anal fissure
37 -> anal fistula
38 -> anemia
39 -> anemia due to chronic kidney 

In [5]:
model = DecisionTreeClassifier()
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print("Accuracy:", accuracy)

Accuracy: 0.8138451881998016


In [6]:
joblib.dump(model, "disease_model.pkl")
joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']

In [7]:
joblib.dump(X.columns.tolist(), "symptoms.pkl")

['symptoms.pkl']

In [9]:
model = joblib.load("disease_model.pkl")
le = joblib.load("label_encoder.pkl")
symptoms = joblib.load("symptoms.pkl")

symptoms = [s.strip().lower() for s in symptoms]

def predict_disease(selected_symptoms):
    selected_symptoms = [s.strip().lower() for s in selected_symptoms]

    user_dict = {symptom: 0 for symptom in symptoms}

    for symptom in selected_symptoms:
        if symptom in user_dict:
            user_dict[symptom] = 1

    input_df = pd.DataFrame([user_dict], columns=symptoms)
    pred = model.predict(input_df)[0]

    return le.inverse_transform([pred])[0]

with gr.Blocks() as demo:
    gr.Markdown("## Disease Predictor")

    gr.Markdown("###  Search Symptoms")
    dropdown = gr.Dropdown(
        choices=symptoms,
        multiselect=True,
        label="Search symptoms"
    )

    gr.Markdown("### Select Symptoms")
    checkbox = gr.CheckboxGroup(
        choices=symptoms,
        label="All symptoms"
    )

    btn = gr.Button("Predict Disease")
    output = gr.Textbox(label="Prediction")

    def combined_predict(d, c):
        selected = list(set((d or []) + (c or [])))
        return predict_disease(selected)

    btn.click(combined_predict, inputs=[dropdown, checkbox], outputs=output)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://78941ad98013cb7455.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
